In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import emcee 
from scipy import signal
from scipy.stats import exponnorm
from scipy.integrate import quad
import pandas as pd
from datetime import datetime
from astropy.timeseries import TimeSeries, aggregate_downsample
from astropy import units as u


In [ ]:
#Parameters 
x = np.arange(-10e-3, 10e-3, 1e-3)

mu = 0 
#sigma = 0.00194912
sigma = 50e-6
tau = 26e-6
Speak = 5e+3

profile = exponnorm.pdf(x, K=tau/sigma, loc=mu, scale=sigma) #Convolution of a gaussian with exponential 
profile *= Speak / profile.max() 
area = np.trapezoid(profile, x)
print(f"Area: {area}")


plt.figure()
plt.plot(x, profile, label='Stokes I pulse')
plt.show()



In [ ]:
x_bb = np.arange(-10e-3, 10e-3, 2.56e-6)

sigma_bb = 50e-6 #50us  intrinsic width
profile_bb = exponnorm.pdf(x_bb, K=tau/sigma_bb, loc=mu, scale=sigma_bb)

profile_bb_norm = profile_bb / profile_bb.max()  # normalized shape, peak = 1
Speak_bb = area / np.trapezoid(profile_bb_norm, x_bb)  # flux needed to match fluence
profile_bb = profile_bb_norm * Speak_bb

area_bb = np.trapezoid(profile_bb, x_bb)
print(f"Area of baseband:{area_bb}")
print(f"New peak flux: {Speak_bb}")

plt.figure()
plt.plot(x_bb, profile_bb, label='Baseband pulse')
plt.show()

In [ ]:
factor = int(1e-3 / 2.56e-6)  # ~390
# trim to multiple of factor
n = len(profile_bb) - len(profile_bb) % factor
downsampled_manual = profile_bb[:n].reshape(-1, factor).mean(axis=1)
x_ds = x_bb[:n].reshape(-1, factor).mean(axis=1)
print(f"Downsampled peak flux: {downsampled_manual.max()}")

plt.figure()
plt.plot(x_ds, downsampled_manual)
plt.show()